In [1]:
SEGMENTATION_PROMPT = """
You are converting a math word problem into a lossless structured representation.

Your goal is NOT to simplify or summarize.
Your goal is to preserve ALL information in atomic form.

Output:
{"segments": [{"segment": "..."}]}

STRICT RULES:

1. Lossless transformation (CRITICAL)
- Every piece of information in the original problem must be preserved.
- Do NOT drop:
  - descriptive context (e.g., preferences, narrative setup)
  - grouping structure
  - qualifiers (e.g., "each", "remaining", "per", "average")

2. Atomic but structure-preserving
- Each segment = ONE fact
- BUT do NOT destroy structure:
  Bad: "third bag = 8", "fourth bag = 8"
  Good:
    "On the third and fourth bags, there are 8 brown M&M's in each bag"

3. Preserve relational language exactly
- Keep words like:
  - "each"
  - "per"
  - "times"
  - "of the remaining"
  - "average"
- These are mathematically critical

4. Preserve implicit variables
- If the problem refers to an unknown quantity, explicitly include it:
  e.g., "Walter spends a certain amount of time looking at the seals"

5. No reasoning or calculation
- Do NOT compute totals or derive values

6. Complete sentence requirement
- Each segment must be a full declarative sentence

7. Question last
- The final segment must restate the question

8. Solvability guarantee
- The segments must be sufficient to reconstruct the original problem EXACTLY

9. No semantic weakening
- Do NOT rewrite in a way that loses constraints
  Example:
  "20 members"
  "each of the 20 members"

Goal:
This is a structure-preserving decomposition, not a simplification.
"""

In [2]:
REPHRASING_PROMPT = """
You are simulating a user revealing a math problem step-by-step in conversation.

Output:
{"shards": [{"shard_id": 1, "shard": "..."}]}

RULES:

1. Question first (anchor)
- shard_id 1 must be the question

2. One-to-one mapping
- Each segment → exactly one shard

3. Preserve ALL meaning
- Do NOT simplify or weaken constraints
- Keep words like "each", "average", "remaining"

4. Conversational but precise
- Natural tone, but mathematically exact

5. Ordering by reasoning importance
- After the question:
  - totals / global constraints first
  - then relationships
  - then details

6. No information drop
- Even narrative context should be included

Goal:
The shards should simulate a user gradually providing all information needed to solve the problem, without losing precision.
"""

In [3]:
VERIFICATION_PROMPT = """
You are verifying whether a set of shards fully preserves a math problem.

Output:
{"coverage": "complete"}
OR
{"coverage": "incomplete", "missing_segment": "..."}

CRITERIA:

1. Information completeness
- All facts must be present

2. Structural completeness (CRITICAL)
- Check if grouping and structure are preserved:
  - "each"
  - "per"
  - "of the remaining"
  - "average"
  - multi-entity relations

3. No semantic weakening
- If a shard weakens meaning → incomplete
  Example:
  "each of the 20 members" → "20 members" → incomplete

4. Implicit variable check
- Missing unknown variable definitions → incomplete

5. Solvability
- The problem must be solvable without guessing

6. Question presence
- Must include the exact goal

Goal:
Reject anything that is not a lossless representation of the original problem.
"""

In [4]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from typing_extensions import NotRequired
import operator
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.types import interrupt, Command, RetryPolicy

/home/jie/anaconda3/envs/attention-tracker/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [13]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="Qwen/Qwen2.5-32B-Instruct",
    openai_api_key="EMPTY",  # vLLM 不校验
    openai_api_base="http://localhost:5001/v1",
    temperature=0.7,
    max_tokens=512,
)

In [6]:
class State(TypedDict):
    dataset_iteration: int
    number_for_generation: int
    instruction: str
    answer: str          # 新增：GSM8K的答案
    task_id: str         # 新增：如 "sharded-GSM8K/14"
    next_task_id: int
    seen_questions: list[str]
    segments: str
    rephrased_output: str
    verification_result: str
    results: list        # 新增：收集所有已完成的样本


In [7]:
def segment_instruction(state: State):
    current_question = _normalize_question(state["instruction"])
    seen_questions = set(state.get("seen_questions", []))

    if current_question in seen_questions:
        print(f"Skipping duplicate question: {state['instruction']}")
        return _next_candidate_update(
            state,
            state["dataset_iteration"] + 1,
            "duplicate question",
        )

    messages = [
        SystemMessage(content=SEGMENTATION_PROMPT),
        HumanMessage(content=state["instruction"])
    ]
    msg = llm.invoke(messages)
    return {"segments": msg.content}


In [8]:
def rephrase_segments(state: State):
    messages = [
        SystemMessage(content=REPHRASING_PROMPT),
        HumanMessage(content=f"Full Query: {state['instruction']} Segments: {state['segments']}"),
    ]
    msg = llm.invoke(messages)
    # print(msg.content)
    return {"rephrased_output": msg.content}

In [9]:
import json
import os

OUTPUT_FILE = "data/(v2)sharded_gsm8k.json"
MIN_SHARDS = 7
MAX_SHARDS = 7
COVERAGE_COMPLETE = "complete"
COVERAGE_INCOMPLETE = "incomplete"


def _load_existing_records() -> list:
    if not os.path.exists(OUTPUT_FILE):
        return []
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            return []
    return data if isinstance(data, list) else []


def _normalize_question(question: str) -> str:
    return " ".join(question.split())


def _get_existing_questions() -> list[str]:
    questions = []
    for record in _load_existing_records():
        question = record.get("question")
        if question:
            questions.append(_normalize_question(question))
    return questions


def _get_next_task_id() -> int:
    max_task_id = -1
    for record in _load_existing_records():
        task_id = str(record.get("task_id", ""))
        try:
            max_task_id = max(max_task_id, int(task_id.rsplit("/", 1)[-1]))
        except ValueError:
            continue
    return max_task_id + 1


def save_record(record: dict):
    """将单条record append写入JSON数组文件"""
    data = _load_existing_records()
    data.append(record)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)


def _parse_shards(rephrased_output: str) -> list:
    """从LLM输出中提取shards"""
    try:
        shards_data = json.loads(rephrased_output)
        return shards_data.get("shards", [])
    except (json.JSONDecodeError, TypeError) as e:
        print(f"Warning: Failed to parse shards: {e}")
        return []


def _next_candidate_update(
    state: State,
    start_i: int,
    verification_result: str = "",
    saved_count: int | None = None,
    seen_questions: list[str] | None = None,
    next_task_id: int | None = None,
) -> Command[Literal["segment_instruction", END]]:
    """找到下一个未生成过的问题；如果没有就结束。"""
    current_saved_count = state.get("saved_count", 0) if saved_count is None else saved_count
    current_seen_questions = state.get("seen_questions", []) if seen_questions is None else seen_questions
    current_next_task_id = state["next_task_id"] if next_task_id is None else next_task_id

    if current_saved_count >= state["number_for_generation"]:
        return Command(
            update={
                "verification_result": verification_result,
                "saved_count": current_saved_count,
                "seen_questions": current_seen_questions,
                "next_task_id": current_next_task_id,
            },
            goto=END,
        )

    seen_question_set = set(current_seen_questions)
    train_split = ds["train"]
    for next_i in range(start_i, len(train_split)):
        next_item = train_split[next_i]
        normalized_question = _normalize_question(next_item["question"])
        if normalized_question in seen_question_set:
            continue

        return Command(
            update={
                "verification_result": verification_result,
                "dataset_iteration": next_i,
                "instruction": next_item["question"],
                "answer": next_item["answer"],
                "task_id": f"sharded-GSM8K/{current_next_task_id}",
                "segments": "",
                "rephrased_output": "",
                "saved_count": current_saved_count,
                "seen_questions": current_seen_questions,
                "next_task_id": current_next_task_id,
            },
            goto="segment_instruction",
        )

    return Command(
        update={
            "verification_result": verification_result,
            "saved_count": current_saved_count,
            "seen_questions": current_seen_questions,
            "next_task_id": current_next_task_id,
        },
        goto=END,
    )


def _handle_valid_shards(state: State, shards: list, verification_result: str):
    """处理验证成功且shard数量有效的情况"""
    n_shards = len(shards)
    print(f"Shard count: {n_shards}")

    if not (MIN_SHARDS <= n_shards <= MAX_SHARDS):
        print(f"Rejected: shard count {n_shards} out of range [{MIN_SHARDS}, {MAX_SHARDS}], retrying...")
        return _next_candidate_update(
            state,
            state["dataset_iteration"] + 1,
            f"rejected: {n_shards} shards",
        )

    current_question = _normalize_question(state["instruction"])
    current_seen_questions = state.get("seen_questions", [])
    if current_question in set(current_seen_questions):
        print(f"Rejected duplicate question: {state['instruction']}")
        return _next_candidate_update(
            state,
            state["dataset_iteration"] + 1,
            "duplicate question",
        )

    # 保存有效的记录
    record = {
        "question": state["instruction"],
        "answer": state["answer"],
        "task_id": state["task_id"],
        "shards": shards,
        "task": "math",
    }
    save_record(record)
    print(f"Saved: {state['task_id']}")

    next_seen_questions = current_seen_questions + [current_question]
    next_saved_count = state.get("saved_count", 0) + 1
    next_task_id = state["next_task_id"] + 1

    if next_saved_count >= state["number_for_generation"]:
        return Command(
            update={
                "saved_count": next_saved_count,
                "seen_questions": next_seen_questions,
                "next_task_id": next_task_id,
                "verification_result": verification_result,
            },
            goto=END,
        )

    return _next_candidate_update(
        state,
        state["dataset_iteration"] + 1,
        verification_result,
        saved_count=next_saved_count,
        seen_questions=next_seen_questions,
        next_task_id=next_task_id,
    )


def verification(state: State) -> Command[Literal["segment_instruction", END]]:
    """验证shards的完整性和数量"""
    messages = [
        SystemMessage(content=VERIFICATION_PROMPT),
        HumanMessage(content=f"Problem: {state['instruction']} Shards: {state['rephrased_output']}"),
    ]
    msg = llm.invoke(messages)
    verification_result = msg.content
    print("Verification result:", verification_result)

    if COVERAGE_COMPLETE in verification_result:
        shards = _parse_shards(state["rephrased_output"])
        return _handle_valid_shards(state, shards, verification_result)

    if COVERAGE_INCOMPLETE in verification_result:
        return _next_candidate_update(
            state,
            state["dataset_iteration"] + 1,
            verification_result,
        )

    return Command(
        update={"verification_result": verification_result},
        goto=END,
    )


In [10]:
workflow = StateGraph(State)

workflow.add_node("segment_instruction", segment_instruction)
workflow.add_node("rephrase_segments", rephrase_segments)
workflow.add_node("verification", verification)

workflow.add_edge(START, "segment_instruction")
workflow.add_edge("segment_instruction", "rephrase_segments")
workflow.add_edge("rephrase_segments", "verification")

# verification 现在通过 Command 显式决定继续下一条还是结束

chain = workflow.compile()


In [11]:
from datasets import load_dataset

random_seed = 43
ds = load_dataset("openai/gsm8k", "socratic")
ds = ds.shuffle(seed=random_seed)
number_for_generation = 20

In [14]:
existing_records = _load_existing_records()
existing_questions = _get_existing_questions()
next_task_id_start = _get_next_task_id()

first_item = ds['train'][0]
state = chain.invoke({
    "dataset_iteration": 0,
    "number_for_generation": number_for_generation,
    "instruction": first_item["question"],
    "answer": first_item["answer"],
    "task_id": f"sharded-GSM8K/{next_task_id_start}",
    "next_task_id": next_task_id_start,
    "seen_questions": existing_questions,
    "saved_count": 0,
    "results": [],
})


Verification result: {"coverage": "incomplete", "missing_segment": "How much did she make over three months, assuming each month is 30 days long?"}

Explanation:
The current shards contain all the necessary information for solving the problem but the question part in Shard 1 is not precise enough. It should explicitly state "assuming each month is 30 days long" to match the original problem's exact phrasing and ensure no ambiguity in the problem statement.
Shard count: 6
Rejected: shard count 6 out of range [7, 7], retrying...
Verification result: {"coverage": "complete"}
Shard count: 4
Rejected: shard count 4 out of range [7, 7], retrying...
Verification result: {"coverage": "incomplete", "missing_segment": "The specific detail about putting 6 candles on each of the remaining cakes is spread across multiple shards (shard 1 and shard 5) but the direct connection and the final question are not grouped together in a single shard, which breaks the structural completeness."}
Shard count: 6

KeyboardInterrupt: 

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-14B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 用你实验里典型的一条messages
messages = [...]  # 粘贴一个真实的例子

from model import _apply_chat_template
prompt = _apply_chat_template(messages, tokenizer)
inputs = tokenizer(prompt, return_tensors="pt")

print(f"sequence length: {inputs['input_ids'].shape[1]} tokens")
print(f"GPU 1 before load: {torch.cuda.memory_allocated(1)/1e9:.2f} GB")

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map={"": "cuda:1"}
)
model.eval()

print(f"GPU 1 after load: {torch.cuda.memory_allocated(1)/1e9:.2f} GB")

with torch.no_grad():
    out = model(**inputs.to("cuda:1"), output_hidden_states=True, return_dict=True)

print(f"GPU 1 after forward: {torch.cuda.memory_allocated(1)/1e9:.2f} GB")
print(f"GPU 1 peak: {torch.cuda.max_memory_allocated(1)/1e9:.2f} GB")